In [24]:
import pandas as pd
import numpy as np

res_hypoxia_noNit = pd.read_csv("/home/nanopore/projects/rna_seq_workflow/results/20250603_FCOM/functional_annotations/MOK_vs_OK_combined_annotations.csv")
res_hypoxia_withNit = pd.read_csv("/home/nanopore/projects/rna_seq_workflow/results/20250603_FCOM/functional_annotations/MOT_vs_OT_combined_annotations.csv")

res_hypoxia_noNit = res_hypoxia_noNit[["symbol", "log2FoldChange", "padj", "KEGG_ko"]]
res_hypoxia_withNit = res_hypoxia_withNit[["symbol", "log2FoldChange", "padj", "KEGG_ko"]]

res_hypoxia_noNit = res_hypoxia_noNit.set_index("symbol")
res_hypoxia_withNit = res_hypoxia_withNit.set_index("symbol")

sig_hypoxia_noNit = res_hypoxia_noNit[
                    (res_hypoxia_noNit['padj'] < 0.05) &
                    (res_hypoxia_noNit['log2FoldChange'].abs() > 1)
                    ]

sig_hypoxia_withNit = res_hypoxia_withNit[
                    (res_hypoxia_withNit['padj'] < 0.05) &
                    (res_hypoxia_withNit['log2FoldChange'].abs() > 2)
                    ]

nitrite_specific = sig_hypoxia_withNit.index.difference(sig_hypoxia_noNit.index)

print(sig_hypoxia_withNit.index)

final_genes = res_hypoxia_withNit.loc[nitrite_specific]

final_genes = final_genes.sort_values(by="log2FoldChange", ascending=False)

final_kos = final_genes.reset_index()
final_kos['KEGG_ko'] = final_kos['KEGG_ko'].str.split(',')
df_exploded = final_kos.explode('KEGG_ko')
df_exploded['KEGG_ko'] = df_exploded['KEGG_ko'].str.strip()
df_exploded = df_exploded.replace('-', np.nan).dropna()
final_kos = df_exploded[["KEGG_ko"]]



# Create a new DataFrame with KEGG IDs and their corresponding log2FoldChange
color_df = df_exploded[["KEGG_ko","log2FoldChange"]].set_index("KEGG_ko").dropna()

final_kos.to_csv("/home/nanopore/projects/rna_seq_workflow/results/20250603_FCOM/functional_annotations/nitrite_specific_kos_exploded.csv", sep="\t", header=None)
color_df.to_csv("/home/nanopore/projects/rna_seq_workflow/results/20250603_FCOM/functional_annotations/nitrite_specific_kos.tsv", sep="\t", header=None)
final_genes.to_csv("/home/nanopore/projects/rna_seq_workflow/results/20250603_FCOM/functional_annotations/nitrite_specific_genes.csv")

Index(['g1201', 'g6087', 'g6087', 'g2509', 'g2277', 'g9125', 'g13681',
       'g13681', 'g12942', 'g8187',
       ...
       'g9787', 'g6033', 'g304', 'g2703', 'g5880', 'g8373', 'g9239', 'g13212',
       'g12936', 'g9386'],
      dtype='object', name='symbol', length=789)
